# 01 — Preprocessing & EDA (Beijing Multi-Site Air Quality)
Mục tiêu: tải dữ liệu, làm sạch, tạo nhãn phân lớp (AQI class theo PM2.5 24h mean), tạo đặc trưng thời gian + lag, và lưu `data/processed/cleaned.parquet`.

**Lưu ý:** nếu `USE_UCIMLREPO=True` thì notebook cần internet để tải dataset từ UCI.

In [ ]:
# [Cell 1]
# Parameters (Papermill sẽ ghi đè cell này)
import sys
import os
from pathlib import Path

# Lấy đường dẫn tuyệt đối của thư mục hiện tại (notebooks/)
current_dir = Path(os.getcwd())

# Tìm thư mục gốc của dự án (nơi chứa folder src và notebooks)
# Logic: Đi ngược lên 1 cấp từ thư mục notebooks
project_root = current_dir.parent

# Thêm thư mục gốc vào sys.path để Python nhìn thấy folder 'src'
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Thử import để kiểm tra ngay lập tức
try:
    from src.classification_library import Paths
    print(f"✅ Đã import thành công src từ: {project_root}")
except ImportError as e:
    print(f"❌ Vẫn lỗi import: {e}")
    # Fallback cho trường hợp chạy local khác cấu trúc
    sys.path.append(os.path.abspath(".."))
USE_UCIMLREPO = False
RAW_ZIP_PATH = "../data/raw/PRSA2017_Data_20130301-20170228.zip"
OUTPUT_CLEANED_PATH = "../data/processed/cleaned.parquet"
LAG_HOURS = [1, 3, 24]

# [Cell 2]
import sys
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Thêm thư mục src vào path để import được library
sys.path.append(os.path.abspath(".."))
from src.classification_library import run_processing_pipeline

# [Cell 3]
# 1. CHẠY PIPELINE XỬ LÝ DỮ LIỆU
# Hàm này sẽ load zip -> clean -> tạo time features -> lưu parquet
run_processing_pipeline(
    raw_zip_path=RAW_ZIP_PATH,
    processed_output_path=OUTPUT_CLEANED_PATH,
    lag_hours=LAG_HOURS
)

# [Cell 4]
# 2. EDA: TRỰC QUAN HÓA DỮ LIỆU (Trả lời Q1)
# Load lại dữ liệu đã sạch để vẽ
df = pd.read_parquet(OUTPUT_CLEANED_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])

# Chọn trạm Aotizhongxin làm mẫu
station_name = "Aotizhongxin"
df_station = df[df['station'] == station_name].sort_values('datetime')

# Vẽ biểu đồ PM2.5 toàn giai đoạn
plt.figure(figsize=(15, 6))
sns.lineplot(data=df_station, x='datetime', y='PM2.5', linewidth=0.5, color='steelblue')
plt.title(f"Diễn biến nồng độ PM2.5 tại trạm {station_name} (2013-2017)")
plt.ylabel("PM2.5 (µg/m³)")
plt.xlabel("Thời gian")
plt.grid(True, alpha=0.3)
plt.show()

# Vẽ biểu đồ Zoom cận cảnh (1 tháng)
start_date = "2016-12-01"
end_date = "2017-01-01"
mask = (df_station['datetime'] >= start_date) & (df_station['datetime'] < end_date)
plt.figure(figsize=(15, 6))
sns.lineplot(data=df_station[mask], x='datetime', y='PM2.5', marker='o', markersize=4, color='darkred')
plt.title(f"Chi tiết PM2.5 tháng 12/2016 ({station_name})")
plt.grid(True, alpha=0.3)
plt.show()